# 04 — Concatenate & Clean New Scopus Data

Combines Lehigh, Marquette, and Villanova Scopus exports into a single cleaned CSV.

**Input files**
```
data/raw/lehigh.csv
data/raw/marquette.csv
data/raw/villanova.csv
```

**Output**
```
data/processed/merged_data.csv
```

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

RAW_DIR  = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Raw Files

In [ ]:
files = {
    'Lehigh':     RAW_DIR / 'lehigh.csv',
    'Marquette':  RAW_DIR / 'marquette.csv',
    'Villanova':  RAW_DIR / 'villanova.csv',
}

frames = []
for institution, path in files.items():
    df = pd.read_csv(path, low_memory=False)
    df['institution'] = institution
    frames.append(df)
    print(f'{institution}: {df.shape[0]:,} rows  |  {df.shape[1]} cols')

print('\nAll files loaded.')

## 3. Inspect Columns (full list from Lehigh as reference)

In [ ]:
# Use Lehigh as the reference schema (all three should be identical Scopus exports)
ref = frames[0]
print(f'Total columns: {len(ref.columns)}\n')
for i, c in enumerate(ref.columns, 1):
    print(f'  {i:3d}.  {c}')

## 4. Feature Checklist

Check whether the features we need for modelling are present.

In [ ]:
# Candidate column names as they commonly appear in Scopus exports.
# Each entry is (friendly_name, [list of possible column name substrings to search for])
FEATURE_MAP = [
    ('Topic Prominence Percentile',  ['topic prominence percentile', 'prominence percentile']),
    ('SNIP',                         ['snip']),
    ('SNIP percentile',              ['snip percentile']),
    ('CiteScore',                    ['citescore']),
    ('CiteScore percentile',         ['citescore percentile']),
    ('SJR',                          ['sjr']),
    ('SJR percentile',               ['sjr percentile']),
    ('Abstract (TF-IDF)',            ['abstract']),
    ('Authors',                      ['authors']),
    ('Author count',                 ['number of authors', 'author count']),
    ('Affiliations',                 ['affiliations', 'affiliation']),
    ('Countries',                    ['country', 'countries']),
    ('Publication year',             ['year']),
    ('Document type (Article/Review)', ['document type', 'document type']),
    ('Source type (Journal/Conf.)',  ['source type']),
    ('Open access status',           ['open access']),
    ('ASJC field classification',    ['asjc', 'all science journal classification']),
    ('Citation count (TARGET)',      ['cited by', 'citations']),
]

cols_lower = {c.lower(): c for c in ref.columns}

results = []
for friendly, patterns in FEATURE_MAP:
    matched = []
    for col_l, col_orig in cols_lower.items():
        if any(p in col_l for p in patterns):
            matched.append(col_orig)
    status = '✅ FOUND' if matched else '❌ MISSING'
    results.append({'Feature': friendly, 'Status': status, 'Matched columns': matched})

checklist = pd.DataFrame(results)
print(checklist.to_string(index=False))

## 5. Concatenate

In [ ]:
raw = pd.concat(frames, ignore_index=True)
print(f'Combined shape: {raw.shape}')
print(f'\nRows per institution:')
print(raw['institution'].value_counts())

## 6. Standardise Column Names

Strip whitespace and normalise the column names so downstream code is consistent.

In [ ]:
raw.columns = raw.columns.str.strip()

# Map the most commonly mis-named Scopus columns to canonical names.
# Edit this dict if your export uses different names.
RENAME = {
    'Cited by':               'Citations',
    'Source title':           'Scopus Source title',
    'Document Type':          'Document type',
    'Publication Stage':      'Publication stage',
    'Open Access':            'Open access',
}

# Only rename columns that actually exist
rename_actual = {k: v for k, v in RENAME.items() if k in raw.columns}
raw.rename(columns=rename_actual, inplace=True)
if rename_actual:
    print('Renamed:', rename_actual)
else:
    print('No renames needed — column names already canonical.')

print(f'\nFinal column list:')
for c in raw.columns:
    print(f'  {c}')

## 7. Missing-Value Audit

In [ ]:
miss = raw.isnull().sum().rename('n_missing')
miss_pct = (miss / len(raw) * 100).round(1).rename('pct_missing')
audit = pd.concat([miss, miss_pct], axis=1)
audit = audit[audit['n_missing'] > 0].sort_values('pct_missing', ascending=False)
print(f'Columns with missing values ({len(audit)} / {len(raw.columns)}):\n')
print(audit.to_string())

## 8. Cleaning

In [ ]:
# ── Year ─────────────────────────────────────────────────────────────────────
year_col = next((c for c in raw.columns if c.lower() == 'year'), None)
if year_col:
    raw[year_col] = pd.to_numeric(raw[year_col], errors='coerce')
    before = len(raw)
    raw = raw[raw[year_col].between(2000, 2026)].copy()
    print(f'Year filter removed {before - len(raw):,} rows  ({len(raw):,} remaining)')
else:
    print('⚠️  No Year column found — skipping year filter')

In [ ]:
# ── Citation count ────────────────────────────────────────────────────────────
cite_col = next((c for c in raw.columns if c.lower() in {'citations', 'cited by'}), None)
if cite_col:
    raw[cite_col] = pd.to_numeric(raw[cite_col], errors='coerce')
    before = len(raw)
    raw = raw[raw[cite_col] >= 0].copy()
    print(f'Citation filter removed {before - len(raw):,} rows  ({len(raw):,} remaining)')
    print(f'Citation count stats:\n{raw[cite_col].describe().round(2)}')
else:
    print('⚠️  No citation column found — please verify column names above')

In [ ]:
# ── Abstract ──────────────────────────────────────────────────────────────────
abs_col = next((c for c in raw.columns if c.lower() == 'abstract'), None)
if abs_col:
    raw[abs_col] = raw[abs_col].astype(str).str.strip()
    # Mark as missing if placeholder text or too short
    placeholder = raw[abs_col].str.lower().isin({'nan', 'none', 'no abstract available', ''})
    too_short   = raw[abs_col].str.len() < 30
    raw.loc[placeholder | too_short, abs_col] = np.nan
    print(f'Abstract present: {raw[abs_col].notna().sum():,} / {len(raw):,}')
else:
    print('⚠️  No Abstract column found')

In [ ]:
# ── Duplicates ────────────────────────────────────────────────────────────────
# Prefer DOI, fall back to Title
doi_col   = next((c for c in raw.columns if c.lower() == 'doi'), None)
title_col = next((c for c in raw.columns if c.lower() == 'title'), None)

before = len(raw)
if doi_col:
    # Drop rows with the same non-null DOI
    mask = raw[doi_col].notna()
    raw_doi   = raw[mask].drop_duplicates(subset=doi_col, keep='first')
    raw_nodoi = raw[~mask]
    raw = pd.concat([raw_doi, raw_nodoi], ignore_index=True)
    print(f'DOI dedup removed {before - len(raw):,} rows')

before2 = len(raw)
if title_col:
    raw[title_col] = raw[title_col].astype(str).str.strip().str.lower()
    raw = raw.drop_duplicates(subset=title_col, keep='first')
    raw[title_col] = raw[title_col].str.title()
    print(f'Title dedup removed {before2 - len(raw):,} additional rows')

print(f'Total remaining: {len(raw):,}')

In [ ]:
# ── Convert numeric venue-metric columns ──────────────────────────────────────
numeric_patterns = ['citescore', 'sjr', 'snip', 'percentile', 'prominence',
                    'number of authors', 'field-weighted']

for col in raw.columns:
    if any(p in col.lower() for p in numeric_patterns):
        raw[col] = pd.to_numeric(raw[col], errors='coerce')

print('Numeric coercion applied to venue-metric columns.')

## 9. Post-Cleaning Summary

In [ ]:
print('=' * 55)
print('CLEANED DATASET SUMMARY')
print('=' * 55)
print(f'Rows    : {len(raw):,}')
print(f'Columns : {len(raw.columns)}')
print(f'Memory  : {raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

print(f'\nRows per institution:')
print(raw['institution'].value_counts())

if year_col:
    print(f'\nYear range : {int(raw[year_col].min())} – {int(raw[year_col].max())}')
if cite_col:
    print(f'Citations  : median={raw[cite_col].median():.0f}  mean={raw[cite_col].mean():.1f}  max={int(raw[cite_col].max())}')

print(f'\nRemaining missing values (top 10):')
miss2 = (raw.isnull().sum() / len(raw) * 100).round(1)
print(miss2[miss2 > 0].sort_values(ascending=False).head(10))

## 10. Re-run Feature Checklist on Cleaned Data

In [ ]:
cols_lower_clean = {c.lower(): c for c in raw.columns}

rows = []
for friendly, patterns in FEATURE_MAP:
    matched = [col_orig for col_l, col_orig in cols_lower_clean.items()
               if any(p in col_l for p in patterns)]
    # Check fill rate for matched columns
    fill_rates = []
    for m in matched:
        fr = raw[m].notna().mean() * 100
        fill_rates.append(f'{m}  ({fr:.0f}% filled)')
    status = '✅ FOUND' if matched else '❌ MISSING'
    rows.append({'Feature': friendly, 'Status': status, 'Column(s) + fill rate': ' | '.join(fill_rates)})

print(pd.DataFrame(rows).to_string(index=False))

## 11. Save

In [ ]:
out_csv = PROC_DIR / 'merged_data.csv'
raw.to_csv(out_csv, index=False)
print(f'✅  Saved  →  {out_csv}  ({len(raw):,} rows)')